# Часть 4 — готовые MCP-серверы

Свой сервер нужен для своих систем. Для всего остального серверы уже написаны — их тысячи,
и подключаются они тем же конфигом, что и наш.

| где искать | что там |
|---|---|
| [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers) | официальные: fetch, filesystem, git, time… |
| [mcp.so](https://mcp.so), [smithery.ai](https://smithery.ai) | каталоги серверов сообщества |
| сами компании | удалённые серверы GitHub, Notion, Atlassian, DeepWiki… |

Подключим два — по одному на каждый транспорт:

- **fetch** — скачивает веб-страницы (локально, stdio, запускается через `uvx`);
- **DeepWiki** — отвечает на вопросы о любом GitHub-репозитории (удалённый, HTTP).

> Чужой сервер — это чужой код или чужой сервис, а то, что он вернул (страница сайта), — недоверенный
> текст. Берите серверы из проверенных источников.

## Шаг 1. fetch — локально через `uvx`

`uvx mcp-server-fetch` сам скачает и запустит пакет. Скачаем официальный курс валют
Чешского национального банка (наш банк тоже чешский):

In [1]:
import shutil

from langchain_mcp_adapters.client import MultiServerMCPClient

fetch_client = MultiServerMCPClient({
    "fetch": {"transport": "stdio", "command": shutil.which("uvx"), "args": ["mcp-server-fetch"]},
})
fetch = (await fetch_client.get_tools())[0]
print(fetch.name, "—", fetch.description[:150], "…")

CNB_RATES = "https://www.cnb.cz/en/financial-markets/foreign-exchange-market/central-bank-exchange-rate-fixing/central-bank-exchange-rate-fixing/daily.txt"
page = (await fetch.ainvoke({"url": CNB_RATES}))[0]["text"]
print("\n".join(line for line in page.splitlines() if "EUR" in line or "USD" in line))

fetch — Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advi …
EMU|euro|1|EUR|24.400
USA|dollar|1|USD|21.811


## Шаг 2. DeepWiki — удалённый, по HTTP

Ничего не устанавливаем — только адрес:

In [2]:
wiki_client = MultiServerMCPClient({
    "deepwiki": {"transport": "streamable_http", "url": "https://mcp.deepwiki.com/mcp"},
})
wiki_tools = {tool.name: tool for tool in await wiki_client.get_tools()}
print(list(wiki_tools))

answer = await wiki_tools["ask_wiki_question"].ainvoke({
    "repoName": "langchain-ai/langchain-mcp-adapters",
    "question": "How do I connect several MCP servers at once? Answer in two sentences.",
})
print(answer[0]["text"][:600])

['ask_wiki_question', 'read_wiki_contents', 'read_wiki_structure']
You can connect to multiple MCP servers simultaneously using the `MultiServerMCPClient` class by providing a dictionary of server configurations during its initialization. This client then allows you to aggregate tools from all connected servers into a unified interface using the `get_tools()` method.  

## Connecting Multiple MCP Servers

To connect several MCP servers at once, you instantiate the `MultiServerMCPClient` class with a dictionary where keys are server names and values are their respective connection parameters . Each server's configuration includes its `transport` type (e.g., "s


## Шаг 3. Один агент — три сервера

Свой `bank` (из части 1) + чужие `fetch` и `deepwiki` — в одном клиенте. Агенту всё равно, откуда
пришли инструменты: он просто видит список и выбирает нужный.

In [3]:
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL", "openai/gpt-4o-mini"), base_url="https://openrouter.ai/api/v1",
                 api_key=os.getenv("OPENROUTER_API_KEY"))

client = MultiServerMCPClient({
    "bank":     {"transport": "stdio", "command": sys.executable, "args": ["db_server.py"]},
    "fetch":    {"transport": "stdio", "command": shutil.which("uvx"), "args": ["mcp-server-fetch"]},
    "deepwiki": {"transport": "streamable_http", "url": "https://mcp.deepwiki.com/mcp"},
})
tools = await client.get_tools()
print([tool.name for tool in tools])

agent = create_agent(llm, tools, system_prompt=(
    "Ты ассистент-аналитик. Отвечай по-русски, кратко и с цифрами. "
    "Банковская база (list_tables, describe_table, run_sql): чешский банк 1993–1998, суммы в кронах (CZK). "
    "fetch — скачать веб-страницу. ask_wiki_question — вопросы про GitHub-репозитории."))


async def ask(question: str) -> None:
    result = await agent.ainvoke({"messages": [("user", question)]})
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                print(f"🤖 → {call['name']}({str(call['args'])[:120]})")
    print("\n💬", result["messages"][-1].text)

['list_tables', 'describe_table', 'run_sql', 'fetch', 'ask_wiki_question', 'read_wiki_contents', 'read_wiki_structure']


In [4]:
print(agent.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | model |           
          +-------+.          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


### Своё + чужое: сумма из базы, курс из интернета

In [5]:
await ask("Какая общая сумма всех кредитов банка? Переведи её в евро по сегодняшнему курсу ЧНБ "
          "(курс возьми через fetch: https://www.cnb.cz/en/financial-markets/foreign-exchange-market/central-bank-exchange-rate-fixing/central-bank-exchange-rate-fixing/daily.txt)")

🤖 → list_tables({})
🤖 → fetch({'url': 'https://www.cnb.cz/en/financial-markets/foreign-exchange-market/central-bank-exchange-rate-fixing/central-bank-)
🤖 → describe_table({'table': 'loans'})
🤖 → run_sql({'sql': 'SELECT COUNT(*) AS loan_count, SUM(amount) AS total_czk FROM loans;'})

💬 Сумма всех 682 кредитов — **103 261 740 CZK**. По курсу ЧНБ **1 EUR = 24,400 CZK** (котировка от 8 октября 2026 года) это **≈ 4 232 038,52 EUR**.

Это сумма выданных кредитов, не остаток задолженности.


In [8]:
f"{103261740 / 24.400:,}"

'4,232,038.524590164'

Проверьте цифры: перевод в евро модель делает «в уме», а LLM в арифметике ошибаются.
Надёжнее попросить посчитать в SQL (`SELECT sum(amount) / 24.43 …`) — считает база, а не модель.

### Вопрос про чужой код

In [9]:
await ask("Как в библиотеке langchain-mcp-adapters передать заголовки авторизации для HTTP-сервера? "
          "Спроси DeepWiki про репозиторий langchain-ai/langchain-mcp-adapters и ответь кратко.")

🤖 → ask_wiki_question({'repoName': 'langchain-ai/langchain-mcp-adapters', 'question': 'How do I pass authorization headers to an HTTP MCP serv)

💬 В конфигурации сервера передайте `headers` — для Bearer-токена укажите заголовок `Authorization`:

```python
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "server": {
        "transport": "http",
        "url": "http://localhost:8000/mcp",
        "headers": {"Authorization": "Bearer YOUR_TOKEN"},
    }
})
```

Поддерживается для HTTP-транспортов (`http`/`streamable_http` и `sse`), не для `stdio`. Для более сложной аутентификации можно передать `auth` с объектом `httpx.Auth`.


## Итоги

- готовые серверы подключаются тем же конфигом: `stdio` + команда или `streamable_http` + адрес;
- один агент держит сколько угодно серверов — модель сама выбирает инструмент;
- чужой сервер = чужой код и недоверенные данные: проверенные источники, минимум прав.

Дальше — часть 5: упаковываем агента в сервис с веб-чатом и памятью в Postgres.